In [1]:
import dalex as dx
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print(f"Dalex version: {dx.__version__}")
print("Arena is built into Dalex — ready to use.")

Dalex version: 1.8.0
Arena is built into Dalex — ready to use.


In [3]:
# Load clean dataset
df = pd.read_csv(r"C:\Users\HP\Documents\ML Project\data\processed\application_train_clean.csv")
X = df.drop(columns=['TARGET'])
y = df['TARGET']

# Remove gender — consistent with deployed model
if 'CODE_GENDER_M' in X.columns:
    X = X.drop(columns=['CODE_GENDER_M'])

# Same split as all previous notebooks
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Load gender-free XGBoost model
import os
models_dir = os.path.join('..', 'models')
xgb_model = joblib.load(
    os.path.join(models_dir, r"C:\Users\HP\Documents\ML Project\loan-default-prediction-\models\xgb_model_no_gender.pkl")
)

# Recreate Logistic Regression for comparison
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    C=0.1
)
lr_model.fit(X_train_scaled, y_train)

print(f"Data: {df.shape}")
print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"XGBoost model loaded.")
print(f"Logistic Regression retrained.")

Data: (307511, 102)
Train: 246,008 | Test: 61,503
XGBoost model loaded.
Logistic Regression retrained.


In [5]:
# Use a sample for speed — ModelStudio can be slow on 61K records
# 3,000 records is sufficient for all visualisations
sample_idx = np.random.choice(
    len(X_test), size=3000, replace=False
)
X_test_sample = X_test.iloc[sample_idx]
y_test_sample = y_test.iloc[sample_idx]

print("Creating Dalex explainers...")

# XGBoost explainer
explainer_xgb = dx.Explainer(
    model=xgb_model,
    data=X_test_sample,
    y=y_test_sample,
    label="XGBoost (Gender-Free)",
    verbose=False
)

# Logistic Regression explainer
X_test_sample_scaled = scaler.transform(X_test_sample)
explainer_lr = dx.Explainer(
    model=lr_model,
    data=pd.DataFrame(
        X_test_sample_scaled,
        columns=X_test_sample.columns
    ),
    y=y_test_sample,
    label="Logistic Regression",
    verbose=False
)

print("Both explainers created.")

Creating Dalex explainers...
Both explainers created.


In [7]:
print("Building Arena dashboard...")
print("This generates all interactive panels.")
print("Takes 3-5 minutes. Do not interrupt.\n")

arena = dx.Arena()
arena.push_model(explainer_xgb)
arena.push_model(explainer_lr)
arena.push_observations(X_test_sample.iloc[:50])

print("Arena dashboard built successfully.")
print(f"\nModels loaded: XGBoost + Logistic Regression")
print(f"Observations: 50 individual borrowers")

Building Arena dashboard...
This generates all interactive panels.
Takes 3-5 minutes. Do not interrupt.

Arena dashboard built successfully.

Models loaded: XGBoost + Logistic Regression
Observations: 50 individual borrowers


In [9]:
import os

output_path = r'C:\Users\HP\Documents\ML Project\loan-default-prediction-\reports\arena_dashboard.html'

print("Saving Arena dashboard...")
print("This may take 2-3 minutes...\n")

# Save the dashboard
arena.save(output_path)

# Confirm file saved and show size
file_size = os.path.getsize(output_path) / 1024 / 1024
print(f"Dashboard saved successfully.")
print(f"File size: {file_size:.1f} MB")
print(f"Location:  {output_path}")
print(f"\nNext steps:")
print(f"1. Open File Explorer")
print(f"2. Navigate to your reports folder")
print(f"3. Double click arena_dashboard.html")
print(f"4. Opens in Chrome or Firefox automatically")
print(f"\nShare this file with your professor")
print(f"— no Python or installation needed to view it.")

Saving Arena dashboard...
This may take 2-3 minutes...



Partial Dependence:   7%|████▎                                                        | 14/200 [00:02<00:30,  6.07it/s]


TypeError: numpy boolean subtract, the `-` operator, is not supported, use the bitwise_xor, the `^` operator, or the logical_xor function instead.

In [ ]:
import os

# Save to reports folder
output_path = '../reports/modelstudio_dashboard.html'

dashboard_comparison.save(
    output_path,
    overwrite=True
)

print(f"Dashboard saved to: {output_path}")
print(f"\nFile size: "
      f"{os.path.getsize(output_path)/1024/1024:.1f} MB")
print(f"\nOpen this file in any browser.")
print(f"Share it with your professor — no Python needed.")
print(f"Share it with the CEO — no installation needed.")

In [ ]:
# Display the dashboard directly inside the notebook
dashboard_comparison